# ALL MAMBA -- Run All 7 Models (Eye Image -> HB Prediction)

Eye fundus/conjunctival image -> Haemoglobin (HB) estimation.
All model code is loaded from **this repo only** -- no external cloning.

| # | Model | Folder | Architecture |
|---|-------|--------|--------------|
| 1 | Mamba1 | 01_Mamba_Official | Selective SSM (Gu & Dao 2023) |
| 2 | Mamba2 | 01_Mamba_Official | Structured State Space (ICML 2024) |
| 3 | Mamba3 | 06_Mamba3_Minimal | MIMO + Rotary SSM (ICLR 2026) |
| 4 | VMamba | 02_VMamba | 2D Cross-Scan SSM (ICLR 2025) |
| 5 | MambaVision | 03_MambaVision | Hybrid Mamba+Transformer (CVPR 2025) |
| 6 | MedMamba | 04_MedMamba | Medical Vision Mamba (2024) |
| 7 | VSSD | 05_VSSD_Mamba2Vision | Vision Mamba-2 (ICCV 2025) |
| 8 | DSA-Mamba | 07_DSA_Mamba_Custom | Series Decomp + Cross-Attention |

> **Edit ONLY Section 1. Run all cells top to bottom.**


## Section 1 -- CONFIGURATION  (Edit only here)

In [ ]:
# ==============================================================
#         ALL VARIABLES -- EDIT ONLY THIS BLOCK
# ==============================================================

# Task:
#   "classification"  ->  Anemic vs Normal   [FocalLoss / weighted CE]
#   "regression"      ->  predict HB g/dL    [Huber / MSE / MAE]
#   Tasks are FULLY SEPARATE -- no combined loss
TASK = "classification"

# Dataset
# DATASET PATHS (Kaggle)
IMAGE_DIR = "/media/junaid/DATA/JUNAID/ANEMIA/dataset/left_eye"
CSV_PATH  = "/media/junaid/DATA/JUNAID/ANEMIA/dataset/merge_excel_1.xlsx"
IMAGE_COL = "Patient ID"   # column whose value IS the image filename (no extension)
HB_COL    = "Haemoglobin (gm/dL)"           # column with hemoglobin measurement (g/dL)
HB_THRESH = 12.0           # below this -> Anemic (label 0), above -> Normal (label 1)

# Training
EPOCHS      = 30
BATCH_SIZE  = 32
LR          = 3e-5          # lower LR avoids early plateau
VAL_SPLIT   = 0.2
NUM_WORKERS = 2
SEED        = 42
EARLY_STOP  = 8             # patience epochs (0 = disabled)
SCHEDULER   = "cosine"      # "cosine" | "plateau" | "none"

# Image size (all models use this)
IMG_SIZE = 224

# ============================================================
# PREPROCESSING -- toggle True/False, nothing else needs to change
# ============================================================
PREPROCESS = {
    # Colorspace: "RGB" | "CIELAB" | "HSV" | "GRAY"
    # CIELAB decouples luminance (L*) from chrominance (a*, b*)
    # -> useful for conjunctival pallor where colour shift is the key signal
    "colorspace"   : "RGB",

    # CLAHE: Contrast-Limited Adaptive Histogram Equalization
    # Boosts local contrast in the L channel before feature extraction
    # Requires OpenCV (pre-installed on Kaggle)
    "use_clahe"    : False,
    "clahe_clip"   : 2.0,      # higher = stronger contrast boost
    "clahe_grid"   : (8, 8),   # tile size for adaptive histogram
}

# ============================================================
# CLASSIFICATION SETTINGS  (only when TASK = "classification")
# ============================================================
CLS_CONFIG = {
    "use_focal_loss"    : True,   # Focal loss -- handles class imbalance
    "focal_gamma"       : 2.0,    # gamma: 0 = CrossEntropy, 2 = standard focal
    "use_class_weights" : True,   # auto-computed from your label distribution
    "threshold"         : 0.5,    # softmax prob threshold for Anemic class
}

# ============================================================
# REGRESSION SETTINGS  (only when TASK = "regression")
# ============================================================
REG_CONFIG = {
    "normalize_hb"  : True,    # scale HB to [0,1] before loss (stabilizes gradients)
    "hb_min"        : 4.0,     # expected dataset min HB (g/dL)
    "hb_max"        : 20.0,    # expected dataset max HB (g/dL)
    "loss_fn"       : "huber", # "mse" | "huber" | "mae"
    "huber_delta"   : 1.0,     # Huber threshold -- beyond this, loss is linear
}

# Which models to run (set False to skip)
RUN = {
    "Mamba1"      : True,
    "Mamba2"      : True,
    "Mamba3"      : True,
    "VMamba"      : True,
    "MambaVision" : True,
    "MedMamba"    : True,
    "VSSD"        : True,
    "DSA_Mamba"   : True,
}

# Paths (auto-detected -- do not change)
import os, sys
REPO_ROOT  = os.getcwd()
MODELS_DIR = os.path.join(REPO_ROOT, "MAMBA_MODELS")
print(f"TASK        : {TASK}")
print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"MODELS_DIR  : {MODELS_DIR}")
print(f"Colorspace  : {PREPROCESS['colorspace']}  |  CLAHE: {PREPROCESS['use_clahe']}")
print(f"Models ON   : {[k for k, v in RUN.items() if v]}")


## Step 0 -- Clone Repo (run once on Kaggle, skip if already cloned)

In [ ]:
import subprocess

REPO_URL  = "https://github.com/junaidmaqbool/ALLMAMBA.git"
CLONE_DIR = "/kaggle/working/ALLMAMBA"

if not os.path.isdir(CLONE_DIR):
    print("Cloning ALLMAMBA ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR], check=True)
    os.chdir(CLONE_DIR)
    REPO_ROOT  = CLONE_DIR
    MODELS_DIR = os.path.join(REPO_ROOT, "MAMBA_MODELS")
    print(f"Cloned. REPO_ROOT = {REPO_ROOT}")
else:
    print("Repo already cloned. Good to go.")


## Section 2 -- Install Dependencies

In [ ]:
import subprocess, sys

PKGS = [
    "mamba-ssm",      # compiled CUDA kernels for Mamba1/2 (Kaggle T4/P100)
    "causal-conv1d",  # required by mamba-ssm
    "einops",
    "timm",
    "pandas",
    "openpyxl",
    "scikit-learn",
    "matplotlib",
    "tqdm",
]
for pkg in PKGS:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True, text=True)
    print(f"  {'OK  ' if r.returncode==0 else 'WARN'} {pkg}")
print("Dependencies done.")


## Section 3 -- Imports & Dataset

In [ ]:
import os, sys, math, time, warnings, traceback
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                              roc_auc_score, mean_absolute_error, f1_score)
from tqdm import tqdm
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}   PyTorch: {torch.__version__}")
print(f"Task   : {TASK}")

# Make MODELS_DIR importable (common/pure_ssm.py lives there)
if MODELS_DIR not in sys.path:
    sys.path.insert(0, MODELS_DIR)


In [ ]:
import numpy as np

# ----------------------------------------------------------------
# Custom colorspace transforms
# ----------------------------------------------------------------
class CLAHETransform:
    """Apply CLAHE to the L channel (Lab space). Requires OpenCV."""
    def __call__(self, img):    # PIL RGB -> PIL RGB
        import cv2
        img_np = np.array(img)
        lab    = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        clahe  = cv2.createCLAHE(clipLimit=PREPROCESS["clahe_clip"],
                                   tileGridSize=PREPROCESS["clahe_grid"])
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))


class ToLABTensor:
    """PIL RGB -> CIELAB tensor (3-ch). L:[0,1]  a*:[-1,1]  b*:[-1,1]"""
    def __call__(self, img):
        import cv2
        img_np = np.array(img).astype(np.uint8)
        lab    = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB).astype(np.float32)
        lab[:, :, 0]  /= 255.0                            # L  : [0, 1]
        lab[:, :, 1]   = (lab[:, :, 1] - 128.0) / 127.0  # a* : [-1, 1]
        lab[:, :, 2]   = (lab[:, :, 2] - 128.0) / 127.0  # b* : [-1, 1]
        return torch.from_numpy(lab.transpose(2, 0, 1))   # (3, H, W)


class ToHSVTensor:
    """PIL RGB -> HSV tensor (3-ch, all channels in [0,1])."""
    def __call__(self, img):
        import cv2
        img_np = np.array(img).astype(np.uint8)
        hsv    = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:, :, 0] /= 179.0   # H: [0, 1]
        hsv[:, :, 1] /= 255.0   # S: [0, 1]
        hsv[:, :, 2] /= 255.0   # V: [0, 1]
        return torch.from_numpy(hsv.transpose(2, 0, 1))   # (3, H, W)


def build_transforms(augment: bool):
    """
    Build the full transform pipeline from PREPROCESS config.
    Change PREPROCESS in Section 1 -- this function adapts automatically.
    Supported colorspaces: RGB | CIELAB | HSV | GRAY
    """
    steps = []

    # 1. Spatial transforms
    if augment:
        steps += [
            transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
            transforms.RandomCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15),
        ]
    else:
        steps.append(transforms.Resize((IMG_SIZE, IMG_SIZE)))

    # 2. CLAHE (applied on PIL before colorspace conversion)
    if PREPROCESS["use_clahe"]:
        steps.append(CLAHETransform())

    # 3. Colorspace conversion + ToTensor + Normalize
    cs = PREPROCESS["colorspace"].upper()
    if cs == "RGB":
        steps += [
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ]
    elif cs == "CIELAB":
        steps.append(ToLABTensor())   # normalisation built in
    elif cs == "HSV":
        steps.append(ToHSVTensor())   # normalisation built in
    elif cs in ("GRAY", "GRAYSCALE"):
        steps += [
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ]
    else:
        raise ValueError(
            f"Unknown colorspace '{PREPROCESS['colorspace']}'. "
            "Choose: RGB | CIELAB | HSV | GRAY")
    return transforms.Compose(steps)


# ----------------------------------------------------------------
# Image-path lookup
# ----------------------------------------------------------------
_IMG_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ""]

def find_image_path(pid: str):
    for ext in _IMG_EXTS:
        p = os.path.join(IMAGE_DIR, str(pid) + ext)
        if os.path.exists(p):
            return p
    return None


# ----------------------------------------------------------------
# Dataset
# ----------------------------------------------------------------
class EyeHBDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        hb    = float(row[HB_COL])
        label = 0 if hb < HB_THRESH else 1

        img = Image.open(row["_img_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)

        return (
            img,
            torch.tensor(label, dtype=torch.long),
            # FIXED: torch.tensor(hb) -> shape () -> (B,) after collate
            # Previously was torch.tensor([[hb]]) -> (B,1,1) which caused
            # silent MSE broadcasting against model output shape (B,1)
            torch.tensor(hb, dtype=torch.float32),
        )


# ----------------------------------------------------------------
# load_data
# ----------------------------------------------------------------
def load_data():
    df = (pd.read_excel(CSV_PATH)
          if CSV_PATH.endswith((".xlsx", ".xls"))
          else pd.read_csv(CSV_PATH))
    total = len(df)

    print(f"Scanning {total} rows for matching image files ...", flush=True)
    df["_img_path"] = df[IMAGE_COL].apply(find_image_path)

    missing_img = df["_img_path"].isna()
    if missing_img.any():
        n = missing_img.sum()
        print(f"  SKIP: {n}/{total} rows -- no image on disk.")
        print(f"  Example missing IDs: {df.loc[missing_img, IMAGE_COL].tolist()[:5]}")
    df = df[~missing_img].copy()

    df[HB_COL] = pd.to_numeric(df[HB_COL], errors="coerce")
    bad_hb = df[HB_COL].isna()
    if bad_hb.any():
        print(f"  SKIP: {bad_hb.sum()} rows with missing/non-numeric HB.")
    df = df[~bad_hb].copy()

    if len(df) == 0:
        raise RuntimeError(
            "Zero valid samples after filtering.\n"
            f"  IMAGE_DIR = {IMAGE_DIR}\n  CSV_PATH  = {CSV_PATH}\n"
            f"  IMAGE_COL = {IMAGE_COL}\n  HB_COL    = {HB_COL}\n"
            "Check that Section 1 paths and column names are correct.")

    print(f"  Valid samples: {len(df)} / {total}")

    # Label distribution -- critical diagnostic
    binary_lb = (df[HB_COL] < HB_THRESH).astype(int)
    n_anemic  = int(binary_lb.sum())
    n_normal  = len(df) - n_anemic
    ratio     = n_anemic / len(df)
    print(f"  Anemic  (HB < {HB_THRESH}): {n_anemic:4d}  ({ratio*100:.1f}%)")
    print(f"  Normal  (HB >= {HB_THRESH}): {n_normal:4d}  ({(1-ratio)*100:.1f}%)")
    print(f"  HB -> mean={df[HB_COL].mean():.2f}  "
          f"min={df[HB_COL].min():.2f}  max={df[HB_COL].max():.2f}  "
          f"std={df[HB_COL].std():.2f} g/dL")
    if abs(ratio - 0.5) > 0.15:
        print(f"  WARNING: Class imbalance ({ratio*100:.0f}% anemic). "
              "CLS_CONFIG use_class_weights=True and use_focal_loss=True recommended.")
    if TASK == "regression":
        print(f"  HB normalise: [{REG_CONFIG['hb_min']}, {REG_CONFIG['hb_max']}] -> [0,1]  "
              f"(enabled={REG_CONFIG['normalize_hb']})")

    tr_df, vl_df = train_test_split(df, test_size=VAL_SPLIT,
                                     random_state=SEED, stratify=binary_lb)

    T_TRAIN = build_transforms(augment=True)
    T_VAL   = build_transforms(augment=False)
    print(f"  Colorspace: {PREPROCESS['colorspace']}  |  CLAHE: {PREPROCESS['use_clahe']}")

    kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
              pin_memory=(DEVICE == "cuda"), drop_last=False)
    tl = DataLoader(EyeHBDataset(tr_df, T_TRAIN), shuffle=True,  **kw)
    vl = DataLoader(EyeHBDataset(vl_df, T_VAL),   shuffle=False, **kw)
    print(f"  Train: {len(tr_df)}  |  Val: {len(vl_df)}")

    # Store for class-weight computation in cell_utils
    global _TRAIN_LABELS
    _TRAIN_LABELS = binary_lb.loc[tr_df.index].tolist()
    return tl, vl


_TRAIN_LABELS            = []
TRAIN_LOADER, VAL_LOADER = load_data()


## Section 4 -- Training Utilities

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# ----------------------------------------------------------------
# HB normalisation helpers (regression only)
# ----------------------------------------------------------------
def _norm_hb(hb: torch.Tensor) -> torch.Tensor:
    lo, hi = REG_CONFIG["hb_min"], REG_CONFIG["hb_max"]
    return (hb - lo) / (hi - lo)

def _denorm_hb(hb_n: torch.Tensor) -> torch.Tensor:
    lo, hi = REG_CONFIG["hb_min"], REG_CONFIG["hb_max"]
    return hb_n * (hi - lo) + lo


# ----------------------------------------------------------------
# Focal Loss
# ----------------------------------------------------------------
class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al. 2017).
    Down-weights easy majority-class examples so the model focuses on
    hard minority-class samples. gamma=0 -> standard CrossEntropy.
    """
    def __init__(self, gamma: float = 2.0, weight: torch.Tensor = None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        ce   = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt   = torch.exp(-ce)               # prob of the true class
        loss = ((1.0 - pt) ** self.gamma) * ce
        return loss.mean()


# ----------------------------------------------------------------
# Build loss functions
# Called at end of this cell, after _TRAIN_LABELS is populated by load_data()
# ----------------------------------------------------------------
def _build_loss_fns():
    global CLS_FN, REG_FN

    if TASK == "classification":
        wt = None
        if CLS_CONFIG["use_class_weights"] and len(_TRAIN_LABELS) > 0:
            w  = compute_class_weight("balanced",
                                       classes=np.array([0, 1]),
                                       y=np.array(_TRAIN_LABELS))
            wt = torch.tensor(w, dtype=torch.float32).to(DEVICE)
            print(f"  Class weights -> Anemic: {wt[0]:.3f}  Normal: {wt[1]:.3f}")

        if CLS_CONFIG["use_focal_loss"]:
            CLS_FN = FocalLoss(gamma=CLS_CONFIG["focal_gamma"], weight=wt)
            print(f"  Loss: FocalLoss  (gamma={CLS_CONFIG['focal_gamma']}, "
                  f"weighted={wt is not None})")
        else:
            CLS_FN = nn.CrossEntropyLoss(weight=wt)
            print(f"  Loss: CrossEntropyLoss  (weighted={wt is not None})")

    elif TASK == "regression":
        fn = REG_CONFIG["loss_fn"].lower()
        if fn == "huber":
            REG_FN = nn.HuberLoss(delta=REG_CONFIG["huber_delta"])
            print(f"  Loss: HuberLoss  (delta={REG_CONFIG['huber_delta']}, "
                  f"normalize_hb={REG_CONFIG['normalize_hb']})")
        elif fn == "mae":
            REG_FN = nn.L1Loss()
            print(f"  Loss: L1Loss (MAE)  (normalize_hb={REG_CONFIG['normalize_hb']})")
        else:
            REG_FN = nn.MSELoss()
            print(f"  Loss: MSELoss  (normalize_hb={REG_CONFIG['normalize_hb']})")
    else:
        raise ValueError(f"TASK must be 'classification' or 'regression', got '{TASK}'")

CLS_FN = None
REG_FN = None
_build_loss_fns()


# ----------------------------------------------------------------
# Compute loss -- fully separated, no combined multi-task loss
# ----------------------------------------------------------------
def compute_loss(logits, hb_pred, labels, hb_true):
    if TASK == "classification":
        return CLS_FN(logits, labels)

    # regression
    hp = hb_pred.view(-1)   # (B,)
    ht = hb_true.view(-1)   # (B,)
    if REG_CONFIG["normalize_hb"]:
        ht = _norm_hb(ht)   # model predicts in [0,1] space
    return REG_FN(hp, ht)


# ----------------------------------------------------------------
# Early Stopper
# ----------------------------------------------------------------
class EarlyStopper:
    def __init__(self, patience: int):
        self.patience   = patience
        self.counter    = 0
        self.best_score = float("inf")

    def step(self, score: float) -> bool:
        """Returns True when training should stop."""
        if score < self.best_score - 1e-6:
            self.best_score = score
            self.counter    = 0
            return False
        self.counter += 1
        return (self.patience > 0) and (self.counter >= self.patience)


# ----------------------------------------------------------------
# run_epoch
# ----------------------------------------------------------------
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    all_preds, all_labels, all_scores = [], [], []
    all_hbp, all_hbt = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels, hb_true in tqdm(loader, leave=False,
                                           desc="train" if training else "val  "):
            imgs, labels, hb_true = (imgs.to(DEVICE),
                                     labels.to(DEVICE),
                                     hb_true.to(DEVICE))
            logits, hb_pred = model(imgs)
            loss = compute_loss(logits, hb_pred, labels, hb_true)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()

            if TASK == "classification":
                probs = torch.softmax(logits, dim=1)
                all_preds.extend(logits.argmax(1).cpu().tolist())
                all_labels.extend(labels.cpu().tolist())
                all_scores.extend(probs[:, 1].cpu().tolist())

            elif TASK == "regression":
                hp = hb_pred.detach().view(-1)
                if REG_CONFIG["normalize_hb"]:
                    hp = _denorm_hb(hp)   # report in real g/dL always
                all_hbp.extend(hp.cpu().tolist())
                all_hbt.extend(hb_true.cpu().view(-1).tolist())

    metrics = {"loss": total_loss / max(len(loader), 1)}

    if TASK == "classification":
        metrics["acc"] = accuracy_score(all_labels, all_preds)
        metrics["f1"]  = f1_score(all_labels, all_preds,
                                   average="macro", zero_division=0)
        try:    metrics["auc"] = roc_auc_score(all_labels, all_scores)
        except: metrics["auc"] = float("nan")

    elif TASK == "regression":
        at, ap = np.array(all_hbt), np.array(all_hbp)
        metrics["mae"]  = float(np.mean(np.abs(at - ap)))
        metrics["rmse"] = float(np.sqrt(np.mean((at - ap) ** 2)))

    return metrics, all_labels, all_preds, all_hbt, all_hbp


# ----------------------------------------------------------------
# run_model
# ----------------------------------------------------------------
RESULTS = []

def run_model(name: str, model: nn.Module):
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    sep    = "=" * 65
    print(f"\n{sep}\n  {name}\n  Params: {params/1e6:.2f}M  |  Task: {TASK}\n{sep}")

    model = model.to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR)

    if SCHEDULER == "cosine":
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    elif SCHEDULER == "plateau":
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(
                  opt, patience=3, factor=0.5, verbose=False)
    else:
        sch = None

    stopper = EarlyStopper(patience=EARLY_STOP)
    hist    = {k: [] for k in ["tr_loss", "vl_loss",
                                "acc", "auc", "f1", "mae", "rmse"]}
    t0      = time.time()
    vl_y = vl_p = None

    for ep in range(1, EPOCHS + 1):
        tr, *_                          = run_epoch(model, TRAIN_LOADER, opt)
        vl, vl_y, vl_p, vl_hbt, vl_hbp = run_epoch(model, VAL_LOADER)

        if sch is not None:
            sch.step(vl["loss"]) if SCHEDULER == "plateau" else sch.step()

        hist["tr_loss"].append(tr["loss"])
        hist["vl_loss"].append(vl["loss"])
        for k in ["acc", "auc", "f1", "mae", "rmse"]:
            hist[k].append(vl.get(k, float("nan")))

        line = f"  Ep {ep:02d}/{EPOCHS}  TL:{tr['loss']:.4f}  VL:{vl['loss']:.4f}"
        if TASK == "classification":
            line += (f"  Acc:{vl.get('acc', 0):.3f}"
                     f"  AUC:{vl.get('auc', 0):.3f}"
                     f"  F1:{vl.get('f1', 0):.3f}")
        elif TASK == "regression":
            line += (f"  MAE:{vl.get('mae', 0):.3f} g/dL"
                     f"  RMSE:{vl.get('rmse', 0):.3f}")
        print(line)

        if stopper.step(vl["loss"]):
            print(f"  Early stop -- no improvement for {EARLY_STOP} epochs.")
            break

    elapsed = time.time() - t0

    if TASK == "classification" and vl_y:
        print("\n  Classification Report (final val epoch):")
        print(classification_report(vl_y, vl_p,
                                     target_names=["Anemic", "Normal"],
                                     zero_division=0))

    last = {k: (v[-1] if v else float("nan")) for k, v in hist.items()}
    RESULTS.append(dict(name=name, status="PASS", params=params,
                        time_s=elapsed, **last, error=""))

    # Learning curves
    ep_x      = list(range(1, len(hist["tr_loss"]) + 1))
    fig, axes = plt.subplots(1, 2, figsize=(12, 3))

    axes[0].plot(ep_x, hist["tr_loss"], label="Train")
    axes[0].plot(ep_x, hist["vl_loss"], label="Val")
    axes[0].set_title(f"{name} -- Loss"); axes[0].legend()

    if TASK == "classification":
        axes[1].plot(ep_x, hist["acc"], label="Acc",  color="green")
        axes[1].plot(ep_x, hist["auc"], label="AUC",  color="purple", linestyle="--")
        axes[1].plot(ep_x, hist["f1"],  label="F1",   color="teal",   linestyle=":")
        axes[1].set_title("Classification Metrics")
        axes[1].legend(); axes[1].set_ylim(0, 1)
    elif TASK == "regression":
        axes[1].plot(ep_x, hist["mae"],  label="MAE (g/dL)",  color="orange")
        axes[1].plot(ep_x, hist["rmse"], label="RMSE (g/dL)", color="red", linestyle="--")
        axes[1].set_title("HB Regression Metrics"); axes[1].legend()

    plt.suptitle(name, fontsize=11, y=1.02)
    plt.tight_layout()
    fname = f"result_{name.split()[0].replace('/', '_')}.png"
    plt.savefig(fname, dpi=100, bbox_inches="tight"); plt.show()
    print(f"  Done in {elapsed:.0f}s  |  chart -> {fname}")
    return model


def _fail(name: str, e: Exception):
    print(f"  {name} FAILED: {e}")
    traceback.print_exc()
    RESULTS.append(dict(name=name, status="FAIL", error=str(e),
                        params=0, time_s=0,
                        tr_loss=float("nan"), vl_loss=float("nan"),
                        acc=float("nan"), auc=float("nan"), f1=float("nan"),
                        mae=float("nan"), rmse=float("nan")))

print("Training utilities ready.")


## Section 5 -- sys.path helper

In [ ]:
def _add(folder: str):
    """Insert a MAMBA_MODELS sub-folder into sys.path (idempotent)."""
    p = os.path.join(MODELS_DIR, folder)
    if p not in sys.path:
        sys.path.insert(0, p)

print("sys.path helper ready.")


---
## Model 1 -- Mamba1 (Official SSM, Gu & Dao 2023)

In [ ]:
if RUN["Mamba1"]:
    _add("01_Mamba_Official")
    try:
        from eye_hb_model import build_mamba1
        mamba1_model = build_mamba1(IMG_SIZE, embed_dim=128, depth=4)
        run_model("Mamba1 (official SSM)", mamba1_model)
    except Exception as e:
        _fail("Mamba1", e)
else:
    print("Mamba1 skipped.")


---
## Model 2 -- Mamba2 / SSD (Dao & Gu, ICML 2024)

In [ ]:
if RUN["Mamba2"]:
    _add("01_Mamba_Official")
    try:
        from eye_hb_model import build_mamba2
        mamba2_model = build_mamba2(IMG_SIZE, embed_dim=128, depth=4)
        run_model("Mamba2 (SSD, ICML 2024)", mamba2_model)
    except Exception as e:
        _fail("Mamba2", e)
else:
    print("Mamba2 skipped.")


---
## Model 3 -- Mamba3 (MIMO + Rotary SSM, ICLR 2026)

In [ ]:
if RUN["Mamba3"]:
    # eye_hb_model for Mamba3 lives in 06_Mamba3_Minimal
    _add("06_Mamba3_Minimal")
    try:
        # Import specifically from the 06 folder
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_m3_eye", os.path.join(MODELS_DIR, "06_Mamba3_Minimal", "eye_hb_model.py"))
        m3_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(m3_eye)
        mamba3_model = m3_eye.build_model(IMG_SIZE, embed_dim=128, depth=4)
        run_model("Mamba3 (ICLR 2026 Oral)", mamba3_model)
    except Exception as e:
        _fail("Mamba3", e)
else:
    print("Mamba3 skipped.")


---
## Model 4 -- VMamba (2D Cross-Scan SSM, ICLR 2025)

In [ ]:
if RUN["VMamba"]:
    _add("02_VMamba")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_vm_eye", os.path.join(MODELS_DIR, "02_VMamba", "eye_hb_model.py"))
        vm_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(vm_eye)
        vmamba_model = vm_eye.build_model(IMG_SIZE)
        run_model("VMamba (2D cross-scan, ICLR 2025)", vmamba_model)
    except Exception as e:
        _fail("VMamba", e)
else:
    print("VMamba skipped.")


---
## Model 5 -- MambaVision (NVIDIA, CVPR 2025)

In [ ]:
if RUN["MambaVision"]:
    _add("03_MambaVision")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_mv_eye", os.path.join(MODELS_DIR, "03_MambaVision", "eye_hb_model.py"))
        mv_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mv_eye)
        mambavision_model = mv_eye.build_model(IMG_SIZE)
        run_model("MambaVision (NVIDIA, CVPR 2025)", mambavision_model)
    except Exception as e:
        _fail("MambaVision", e)
else:
    print("MambaVision skipped.")


---
## Model 6 -- MedMamba (Medical Vision Mamba, 2024)

In [ ]:
if RUN["MedMamba"]:
    _add("04_MedMamba")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_med_eye", os.path.join(MODELS_DIR, "04_MedMamba", "eye_hb_model.py"))
        med_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(med_eye)
        medmamba_model = med_eye.build_model(IMG_SIZE)
        run_model("MedMamba (medical imaging, 2024)", medmamba_model)
    except Exception as e:
        _fail("MedMamba", e)
else:
    print("MedMamba skipped.")


---
## Model 7 -- VSSD / VMAMBA2 (Mamba2 Vision, ICCV 2025)

In [ ]:
if RUN["VSSD"]:
    _add("05_VSSD_Mamba2Vision")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_vssd_eye", os.path.join(MODELS_DIR, "05_VSSD_Mamba2Vision", "eye_hb_model.py"))
        vssd_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(vssd_eye)
        vssd_model = vssd_eye.build_model(IMG_SIZE)
        run_model("VSSD (Mamba2 Vision, ICCV 2025)", vssd_model)
    except Exception as e:
        _fail("VSSD", e)
else:
    print("VSSD skipped.")


---
## Model 9 -- DSA-Mamba Custom (dual-task eye/HB, 2024)

In [ ]:
if RUN["DSA_Mamba"]:
    _add("07_DSA_Mamba_Custom")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_dsa_eye", os.path.join(MODELS_DIR, "07_DSA_Mamba_Custom", "eye_hb_model.py"))
        dsa_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(dsa_eye)
        dsa_model = dsa_eye.build_model(IMG_SIZE)
        run_model("DSA-Mamba Custom (eye HB dual-task, 2024)", dsa_model)
    except Exception as e:
        _fail("DSA_Mamba", e)
else:
    print("DSA-Mamba skipped.")


---
## Results Summary

In [ ]:
if RESULTS:
    df_res = pd.DataFrame(RESULTS)

    base_cols  = ["name", "status", "params", "time_s", "tr_loss", "vl_loss"]
    if TASK == "classification":
        extra = ["acc", "auc", "f1"]
    elif TASK == "regression":
        extra = ["mae", "rmse"]
    else:
        extra = ["acc", "auc", "f1", "mae", "rmse"]
    show_cols = [c for c in base_cols + extra if c in df_res.columns]
    df_show   = df_res[show_cols].copy()

    for col in df_show.select_dtypes(include="number").columns:
        if col == "params":
            df_show[col] = df_show[col].apply(
                lambda x: f"{x/1e6:.2f}M"
                if not (isinstance(x, float) and np.isnan(x)) else "n/a")
        elif col == "time_s":
            df_show[col] = df_show[col].apply(
                lambda x: f"{x:.0f}s"
                if not (isinstance(x, float) and np.isnan(x)) else "n/a")
        else:
            df_show[col] = df_show[col].apply(
                lambda x: f"{x:.4f}"
                if not (isinstance(x, float) and np.isnan(x)) else "n/a")

    print("\n" + "=" * 70)
    print("  FINAL RESULTS")
    print("=" * 70)
    print(df_show.to_string(index=False))

    passed = df_res[df_res["status"] == "PASS"]
    if not passed.empty:
        print()
        if TASK == "classification":
            if "auc" in passed.columns:
                best = passed.loc[passed["auc"].idxmax()]
                print(f"  Best AUC : {best['name']}  AUC={best['auc']:.4f}")
            if "acc" in passed.columns:
                best_a = passed.loc[passed["acc"].idxmax()]
                print(f"  Best Acc : {best_a['name']}  Acc={best_a['acc']:.4f}")
            if "f1" in passed.columns:
                best_f = passed.loc[passed["f1"].idxmax()]
                print(f"  Best F1  : {best_f['name']}  F1={best_f['f1']:.4f}")
        elif TASK == "regression":
            if "mae" in passed.columns:
                best = passed.loc[passed["mae"].idxmin()]
                print(f"  Best MAE : {best['name']}  MAE={best['mae']:.4f} g/dL")
else:
    print("No results -- all models were skipped or failed.")
